# Capítulo 9: Versionando el Caos (DVC y Control de Versiones)

## Objetivo del Notebook
Este notebook demuestra los conceptos prácticos del Capítulo 9:
- Configuración de DVC
- Inicialización de Git y DVC
- Tracking de datos
- Estructura de proyecto reproducible
- Tracking de experimentos con MLflow
- Reproducibilidad
- Ética y trazabilidad

## Celda 1: Configuración de DVC

**Concepto:** DVC (Data Version Control) es el "control de versiones de Google Drive pero profesional". Mientras Git trackea código, DVC trackea datos y modelos.

**Metáfora:** Imagina que Git es como un historial de documentos de texto, y DVC es como un historial de archivos pesados. DVC crea archivos `.dvc` ligeros que contienen metadatos, mientras los datos reales se almacenan en un almacenamiento remoto.

In [ ]:
# Instalar dependencias necesarias
# Descomenta las siguientes líneas si es la primera vez que ejecutas este notebook
# !pip install dvc
# !pip install dvc[s3]  # Si usas S3 como remote
# !pip install dvc[gdrive]  # Si usas Google Drive
# !pip install mlflow
# !pip install scikit-learn pandas pyyaml

import subprocess
import sys

def install_if_missing(package, import_name=None):
    """Instala un paquete si no está disponible."""
    if import_name is None:
        import_name = package
    try:
        __import__(import_name)
        print(f"✓ {package} ya está instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} instalado correctamente")

# Verificar e instalar paquetes
install_if_missing('dvc')
install_if_missing('mlflow')
install_if_missing('pandas')
install_if_missing('sklearn', 'sklearn')
install_if_missing('yaml')
install_if_missing('matplotlib')
install_if_missing('seaborn')

In [ ]:
import os
import json
import yaml
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Verificar versión de DVC
import dvc
print(f"DVC versión: {dvc.__version__}")

# Verificar versión de MLflow
import mlflow
print(f"MLflow versión: {mlflow.__version__}")

# Verificar que Git está disponible
import subprocess
try:
    result = subprocess.run(['git', '--version'], capture_output=True, text=True)
    print(f"Git: {result.stdout.strip()}")
except FileNotFoundError:
    print("⚠ Git no está instalado. Instálalo con: sudo apt-get install git")

print("\n✓ Configuración completada")

## Celda 2: Inicialización de Git y DVC

**Concepto:** Git para código, DVC para datos. El Mandamiento 4 dice: "Datos cambian, modelos se corrompen."

**Flujo:**
1. `git init` → Inicializa repositorio para código
2. `dvc init` → Inicializa DVC para datos
3. Configurar remote storage → Donde se almacenan los datos

In [ ]:
# Crear estructura del proyecto
project_name = "proyecto_ml_cap09"
project_path = Path(project_name)

# Crear directorios
directories = [
    "data/raw",
    "data/processed",
    "src/pipelines",
    "src/experiments",
    "models/saved",
    "metrics",
    "plots",
    "metadata",
    "config",
    "tests"
]

for dir_path in directories:
    (project_path / dir_path).mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {dir_path}")

print(f"\nEstructura creada en: {project_path}")

In [ ]:
# Inicializar Git
import subprocess

def run_command(cmd, cwd=None):
    """Ejecuta un comando y retorna el resultado."""
    result = subprocess.run(
        cmd, 
        shell=True, 
        capture_output=True, 
        text=True,
        cwd=cwd
    )
    return result.returncode, result.stdout, result.stderr

# Inicializar Git
print("=== Inicializando Git ===")
run_command("git init", cwd=str(project_path))
run_command("git config user.email \"ejemplo@ciencia-datos.com\"", cwd=str(project_path))
run_command("git config user.name \"Científico de Datos\"", cwd=str(project_path))
print("✓ Git inicializado")

# Inicializar DVC
print("\n=== Inicializando DVC ===")
returncode, stdout, stderr = run_command("dvc init", cwd=str(project_path))
if returncode == 0:
    print("✓ DVC inicializado")
else:
    print(f"Error: {stderr}")

# Verificar estado
print("\n=== Estado del repositorio ===")
returncode, stdout, stderr = run_command("git status", cwd=str(project_path))
print(stdout)

In [ ]:
# Configurar remote storage (Google Drive como ejemplo)
# NOTA: Para producción, usa S3, Azure Blob, o GCS

print("=== Configurando Remote Storage ===")

# Opción 1: Directorio local (para pruebas)
remote_path = Path("/tmp/dvc_remote_storage")
remote_path.mkdir(exist_ok=True)

returncode, stdout, stderr = run_command(
    f"dvc remote add -d local_remote {remote_path}",
    cwd=str(project_path)
)

if returncode == 0:
    print(f"✓ Remote configurado: {remote_path}")
else:
    print(f"Remote ya existe o error: {stderr}")

# Para Google Drive (descomenta si necesitas)
# returncode, _, _ = run_command(
#     "dvc remote add -d gdrive_remote gdrive://mi_carpeta_ml",
#     cwd=str(project_path)
# )

# Para S3 (descomenta si necesitas)
# returncode, _, _ = run_command(
#     "dvc remote add -d s3_remote s3://mi-bucket/ml-data",
#     cwd=str(project_path)
# )

print("✓ Configuración de remote completada")

## Celda 3: Tracking de Datos

**Concepto:** DVC no almacena datos en Git. Crea archivos `.dvc` ligeros con hashes. Los datos reales van al remote storage.

**Metáfora:** Es como tener un catálogo de biblioteca. El catálogo (archivo .dvc) te dice dónde está el libro (datos), pero el libro no está en el catálogo.

In [ ]:
# Crear dataset sintético para el ejemplo
np.random.seed(42)
n_samples = 1000

data = {
    'customer_id': range(1, n_samples + 1),
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.normal(50000, 15000, n_samples).astype(int),
    'tenure_months': np.random.randint(1, 60, n_samples),
    'usage_score': np.random.uniform(0, 100, n_samples).round(2),
    'gender': np.random.choice(['M', 'F', 'O'], n_samples),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], n_samples),
    'churned': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
}

df = pd.DataFrame(data)

# Guardar dataset
raw_data_path = project_path / "data" / "raw" / "clientes_v1.csv"
df.to_csv(raw_data_path, index=False)

print(f"Dataset creado: {raw_data_path}")
print(f"Registros: {len(df)}")
print(f"Columnas: {list(df.columns)}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Agregar dataset a DVC
print("=== Agregando dataset a DVC ===")

returncode, stdout, stderr = run_command(
    f"dvc add data/raw/clientes_v1.csv",
    cwd=str(project_path)
)

if returncode == 0:
    print("✓ Dataset agregado a DVC")
    print(stdout)
else:
    print(f"Error: {stderr}")

# Verificar archivo .dvc creado
dvc_file = project_path / "data" / "raw" / "clientes_v1.csv.dvc"
if dvc_file.exists():
    print(f"\n✓ Archivo DVC creado: {dvc_file}")
    print("\nContenido del archivo .dvc:")
    print(dvc_file.read_text())

# Verificar .gitignore actualizado
gitignore = project_path / ".gitignore"
if gitignore.exists():
    print(f"\n.gitignore actualizado:")
    print(gitignore.read_text())

In [ ]:
# Simular actualización de datos (v2)
print("=== Creando versión 2 del dataset ===")

# Agregar nuevos registros
n_new = 200
new_data = {
    'customer_id': range(n_samples + 1, n_samples + n_new + 1),
    'age': np.random.randint(18, 70, n_new),
    'income': np.random.normal(55000, 18000, n_new).astype(int),  # Ingresos más altos
    'tenure_months': np.random.randint(1, 36, n_new),
    'usage_score': np.random.uniform(0, 100, n_new).round(2),
    'gender': np.random.choice(['M', 'F', 'O'], n_new),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], n_new),
    'churned': np.random.choice([0, 1], n_new, p=[0.75, 0.25])  # Más churn
}

df_v2 = pd.concat([df, pd.DataFrame(new_data)], ignore_index=True)

# Guardar versión 2
raw_data_v2_path = project_path / "data" / "raw" / "clientes_v2.csv"
df_v2.to_csv(raw_data_v2_path, index=False)

# Agregar a DVC
run_command(
    f"dvc add data/raw/clientes_v2.csv",
    cwd=str(project_path)
)

print(f"Dataset v2 creado: {raw_data_v2_path}")
print(f"Total registros: {len(df_v2)}")
print(f"\nDiferencias entre versiones:")
print(f"  v1: {len(df)} registros, churn rate: {df['churned'].mean():.2%}")
print(f"  v2: {len(df_v2)} registros, churn rate: {df_v2['churned'].mean():.2%}")

In [ ]:
# Push datos al remote storage
print("=== Push de datos al remote storage ===")

returncode, stdout, stderr = run_command(
    "dvc push",
    cwd=str(project_path)
)

if returncode == 0:
    print("✓ Datos enviados al remote storage")
    print(stdout)
else:
    print(f"Error: {stderr}")

# Verificar estado de DVC
print("\n=== Estado de DVC ===")
returncode, stdout, stderr = run_command(
    "dvc status",
    cwd=str(project_path)
)
print(stdout)

## Celda 4: Estructura de Proyecto

**Concepto:** La reproducibilidad es como tener una receta de cocina que siempre funciona. Mismos ingredientes + mismos pasos = mismo resultado.

**Archivos clave:**
- `params.yaml` → Hiperparámetros versionados
- `dvc.yaml` → Pipeline de reprocción
- `requirements.txt` → Dependencias exactas

In [ ]:
# Crear params.yaml
params_config = {
    'data': {
        'raw_path': 'data/raw/clientes_v2.csv',
        'processed_path': 'data/processed/dataset_clean.csv',
        'test_size': 0.2,
        'random_state': 42
    },
    'features': {
        'numeric_cols': ['age', 'income', 'tenure_months', 'usage_score'],
        'categorical_cols': ['gender', 'region'],
        'scaling': 'standard'
    },
    'model': {
        'type': 'random_forest',
        'params': {
            'n_estimators': 100,
            'max_depth': 10,
            'min_samples_split': 5,
            'random_state': 42
        }
    },
    'training': {
        'cv_folds': 5,
        'scoring': 'f1_weighted'
    }
}

params_path = project_path / "config" / "params.yaml"
with open(params_path, 'w') as f:
    yaml.dump(params_config, f, default_flow_style=False)

print("✓ params.yaml creado:")
print(yaml.dump(params_config, default_flow_style=False))

In [ ]:
# Crear pipeline script de preparación de datos
prepare_script = '''
"""Pipeline: Preparación de datos"""
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from pathlib import Path

def prepare_data():
    # Cargar parámetros
    with open('config/params.yaml', 'r') as f:
        params = yaml.safe_load(f)
    
    # Cargar datos
    df = pd.read_csv(params['data']['raw_path'])
    
    # Eliminar duplicados
    df = df.drop_duplicates()
    
    # Eliminar valores nulos
    df = df.dropna()
    
    # Dividir en train/test
    train, test = train_test_split(
        df,
        test_size=params['data']['test_size'],
        random_state=params['data']['random_state'],
        stratify=df['churned']
    )
    
    # Guardar
    Path('data/processed').mkdir(parents=True, exist_ok=True)
    train.to_csv('data/processed/train.csv', index=False)
    test.to_csv('data/processed/test.csv', index=False)
    
    print(f"Train: {len(train)} registros")
    print(f"Test: {len(test)} registros")
    
    return train, test

if __name__ == '__main__':
    prepare_data()
'''

prepare_path = project_path / "src" / "pipelines" / "prepare.py"
prepare_path.write_text(prepare_script)
print(f"✓ Script de preparación creado: {prepare_path}")

In [ ]:
# Crear pipeline script de entrenamiento
train_script = '''
"""Pipeline: Entrenamiento de modelo"""
import pandas as pd
import yaml
import json
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from pathlib import Path

def train_model():
    # Cargar parámetros
    with open('config/params.yaml', 'r') as f:
        params = yaml.safe_load(f)
    
    # Cargar datos procesados
    train = pd.read_csv('data/processed/train.csv')
    test = pd.read_csv('data/processed/test.csv')
    
    # Preparar features
    feature_cols = (
        params['features']['numeric_cols'] + 
        params['features']['categorical_cols']
    )
    
    X_train = pd.get_dummies(train[feature_cols], drop_first=True)
    y_train = train['churned']
    X_test = pd.get_dummies(test[feature_cols], drop_first=True)
    y_test = test['churned']
    
    # Entrenar modelo
    model = RandomForestClassifier(**params['model']['params'])
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred)
    }
    
    # Guardar modelo
    Path('models/saved').mkdir(parents=True, exist_ok=True)
    joblib.dump(model, 'models/saved/model.pkl')
    
    # Guardar métricas
    Path('metrics').mkdir(parents=True, exist_ok=True)
    with open('metrics/train_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print("Métricas de entrenamiento:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")
    
    return model, metrics

if __name__ == '__main__':
    train_model()
'''

train_path = project_path / "src" / "pipelines" / "train.py"
train_path.write_text(train_script)
print(f"✓ Script de entrenamiento creado: {train_path}")

In [ ]:
# Crear dvc.yaml (pipeline de reprocción)
dvc_config = {
    'stages': {
        'prepare': {
            'cmd': 'python src/pipelines/prepare.py',
            'deps': [
                'src/pipelines/prepare.py',
                'data/raw/clientes_v2.csv'
            ],
            'params': [
                'data.test_size',
                'data.random_state'
            ],
            'outs': [
                'data/processed/train.csv',
                'data/processed/test.csv'
            ]
        },
        'train': {
            'cmd': 'python src/pipelines/train.py',
            'deps': [
                'src/pipelines/train.py',
                'data/processed/train.csv',
                'data/processed/test.csv'
            ],
            'params': [
                'model.type',
                'model.params',
                'features.numeric_cols',
                'features.categorical_cols'
            ],
            'outs': [
                'models/saved/model.pkl'
            ],
            'metrics': [
                'metrics/train_metrics.json'
            ]
        }
    }
}

dvc_yaml_path = project_path / "dvc.yaml"
with open(dvc_yaml_path, 'w') as f:
    yaml.dump(dvc_config, f, default_flow_style=False)

print("✓ dvc.yaml creado:")
print(yaml.dump(dvc_config, default_flow_style=False))

In [ ]:
# Crear requirements.txt
requirements = """pandas==2.0.3
numpy==1.24.3
scikit-learn==1.3.0
xgboost==1.7.6
mlflow==2.7.1
dvc==3.23.0
pyyaml==6.0.1
matplotlib==3.7.2
seaborn==0.12.2
joblib==1.3.2
"""

req_path = project_path / "requirements.txt"
req_path.write_text(requirements)
print(f"✓ requirements.txt creado: {req_path}")

In [ ]:
# Crear .gitignore completo
gitignore_content = """
# Datos (versionados con DVC)
data/raw/
data/processed/
*.csv
*.parquet
*.h5

# Modelos
models/saved/
*.pkl
*.joblib
*.h5
*.pt

# Entornos
__pycache__/
*.pyc
.env
venv/

# Jupyter
.ipynb_checkpoints/

# Archivos del sistema
.DS_Store
Thumbs.db

# MLflow
mlruns/
mlartifacts/
"""

gitignore_path = project_path / ".gitignore"
gitignore_path.write_text(gitignore_content)
print(f"✓ .gitignore creado: {gitignore_path}")

In [ ]:
# Ejecutar pipeline con DVC
print("=== Ejecutando pipeline con DVC ===")

returncode, stdout, stderr = run_command(
    "dvc repro",
    cwd=str(project_path)
)

if returncode == 0:
    print("✓ Pipeline ejecutado exitosamente")
    print("\nSalida:")
    print(stdout)
else:
    print(f"Error: {stderr}")
    print("\nSalida estándar:")
    print(stdout)

In [ ]:
# Ver métricas del pipeline
print("=== Métricas del pipeline ===")

returncode, stdout, stderr = run_command(
    "dvc metrics show",
    cwd=str(project_path)
)

print(stdout)

# Verificar archivos creados
print("\n=== Archivos creados ===")
for root, dirs, files in os.walk(project_path):
    level = root.replace(str(project_path), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Mostrar solo primeros 5 archivos
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... y {len(files) - 5} más")

## Celda 5: Tracking de Experimentos con MLflow

**Concepto:** MLflow es como un diario profesional que registra automáticamente cada experimento. Cada ejecución tiene sus parámetros, métricas y modelo.

**Cita:** Zaharia et al. (2018) lo describen como "una plataforma open source para el ciclo de vida completo del machine learning".

In [ ]:
# Script de tracking con MLflow
mlflow_tracking_script = '''
"""Experiment Tracking con MLflow"""
import mlflow
import mlflow.sklearn
import pandas as pd
import yaml
import json
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import cross_val_score
import numpy as np

def load_data():
    """Carga datos procesados."""
    train = pd.read_csv('data/processed/train.csv')
    test = pd.read_csv('data/processed/test.csv')
    
    with open('config/params.yaml', 'r') as f:
        params = yaml.safe_load(f)
    
    feature_cols = (
        params['features']['numeric_cols'] + 
        params['features']['categorical_cols']
    )
    
    X_train = pd.get_dummies(train[feature_cols], drop_first=True)
    y_train = train['churned']
    X_test = pd.get_dummies(test[feature_cols], drop_first=True)
    y_test = test['churned']
    
    # Alinear columnas
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    
    return X_train, X_test, y_train, y_test

def run_experiment(experiment_name, model_class, model_params, X_train, X_test, y_train, y_test):
    """Ejecuta un experimento y registra en MLflow."""
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=f"{model_class.__name__}_run"):
        # Log parámetros
        mlflow.log_params(model_params)
        mlflow.log_param("model_type", model_class.__name__)
        mlflow.log_param("dataset_version", "v2")
        mlflow.log_param("n_samples", len(X_train) + len(X_test))
        
        # Entrenar modelo
        model = model_class(**model_params)
        model.fit(X_train, y_train)
        
        # Predecir
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        # Calcular métricas
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_score': f1_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'roc_auc': roc_auc_score(y_test, y_proba)
        }
        
        # Log métricas
        mlflow.log_metrics(metrics)
        
        # Log modelo
        mlflow.sklearn.log_model(
            model,
            "model",
            registered_model_name=f"{model_class.__name__}_classifier"
        )
        
        print(f"\n{'='*50}")
        print(f"Experimento: {experiment_name}")
        print(f"Modelo: {model_class.__name__}")
        print(f"Métricas:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
        
        return metrics

def main():
    """Ejecuta múltiples experimentos."""
    X_train, X_test, y_train, y_test = load_data()
    
    # Definir experimentos
    experiments = [
        {
            'name': 'random_forest_experiment',
            'class': RandomForestClassifier,
            'params': {
                'n_estimators': 100,
                'max_depth': 10,
                'min_samples_split': 5,
                'random_state': 42
            }
        },
        {
            'name': 'random_forest_experiment',
            'class': RandomForestClassifier,
            'params': {
                'n_estimators': 200,
                'max_depth': 15,
                'min_samples_split': 3,
                'random_state': 42
            }
        },
        {
            'name': 'gradient_boosting_experiment',
            'class': GradientBoostingClassifier,
            'params': {
                'n_estimators': 100,
                'learning_rate': 0.1,
                'max_depth': 5,
                'random_state': 42
            }
        },
        {
            'name': 'logistic_regression_experiment',
            'class': LogisticRegression,
            'params': {
                'C': 1.0,
                'max_iter': 1000,
                'random_state': 42
            }
        }
    ]
    
    results = []
    for exp in experiments:
        metrics = run_experiment(
            exp['name'],
            exp['class'],
            exp['params'],
            X_train, X_test, y_train, y_test
        )
        results.append({
            'model': exp['class__.__name____,
            'params': exp['params'],
            'metrics': metrics
        })
    
    # Guardar resumen
    with open('metrics/experiments_summary.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print(f"\n{'='*50}")
    print("✓ Todos los experimentos completados")
    print("Resumen guardado en: metrics/experiments_summary.json")

if __name__ == '__main__':
    main()
'''

mlflow_script_path = project_path / "src" / "experiments" / "run_experiments.py"
mlflow_script_path.write_text(mlflow_tracking_script)
print(f"✓ Script de MLflow creado: {mlflow_script_path}")

In [ ]:
# Ejecutar experimentos
print("=== Ejecutando experimentos con MLflow ===")

returncode, stdout, stderr = run_command(
    "python src/experiments/run_experiments.py",
    cwd=str(project_path)
)

if returncode == 0:
    print("✓ Experimentos ejecutados")
    print(stdout)
else:
    print(f"Error: {stderr}")
    print("Salida:")
    print(stdout)

# Nota: Para ver la interfaz web de MLflow, ejecuta:
# mlflow ui
# Y abre http://localhost:5000

## Celda 6: Reproducibilidad

**Concepto:** La reproducibilidad es la capacidad de obtener exactamente los mismos resultados ejecutando el mismo código con los mismos datos.

**Metáfora:** Es como una receta de cocina que siempre funciona. Si sigues los pasos exactos, con los ingredientes exactos, siempre obtienes el mismo plato.

In [ ]:
# Demostrar reproducibilidad
print("=== Demostración de Reproducibilidad ===")
print("\n1. Primera ejecución del pipeline:")
print("   (Los datos ya fueron procesados en la Celda 4)")

# Verificar que los archivos existen
train_path = project_path / "data" / "processed" / "train.csv"
test_path = project_path / "data" / "processed" / "test.csv"
model_path = project_path / "models" / "saved" / "model.pkl"
metrics_path = project_path / "metrics" / "train_metrics.json"

print(f"\n2. Verificando archivos generados:")
print(f"   Train: {train_path.exists()} ({len(pd.read_csv(train_path))} registros)")
print(f"   Test: {test_path.exists()} ({len(pd.read_csv(test_path))} registros)")
print(f"   Model: {model_path.exists()}")
print(f"   Metrics: {metrics_path.exists()}")

# Mostrar métricas
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print(f"\n3. Métricas del modelo:")
    for metric, value in metrics.items():
        print(f"   {metric}: {value:.4f}")

In [ ]:
# Simular cambio en hiperparámetros y ver cambio en métricas
print("=== Cambio de Hiperparámetros ===")

# Modificar params.yaml
with open(project_path / "config" / "params.yaml", 'r') as f:
    params = yaml.safe_load(f)

# Guardar métricas originales
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        original_metrics = json.load(f)
else:
    original_metrics = {}

print("\nHiperparámetros originales:")
print(f"  n_estimators: {params['model']['params']['n_estimators']}")
print(f"  max_depth: {params['model']['params']['max_depth']}")

# Modificar hiperparámetros
params['model']['params']['n_estimators'] = 50
params['model']['params']['max_depth'] = 5

with open(project_path / "config" / "params.yaml", 'w') as f:
    yaml.dump(params, f, default_flow_style=False)

print("\nHiperparámetros modificados:")
print(f"  n_estimators: {params['model']['params']['n_estimators']}")
print(f"  max_depth: {params['model']['params']['max_depth']}")

# Ejecutar pipeline nuevamente
print("\n4. Re-ejecutando pipeline con nuevos parámetros:")
returncode, stdout, stderr = run_command(
    "dvc repro",
    cwd=str(project_path)
)

# Comparar métricas
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        new_metrics = json.load(f)
    
    print("\n5. Comparación de métricas:")
    print(f"{'Métrica':<15} {'Original':<12} {'Nueva':<12} {'Diferencia':<12}")
    print("-" * 51)
    for metric in original_metrics:
        if metric in new_metrics:
            diff = new_metrics[metric] - original_metrics[metric]
            print(f"{metric:<15} {original_metrics[metric]:<12.4f} {new_metrics[metric]:<12.4f} {diff:+.4f}")

In [ ]:
# Restaurar parámetros originales
print("=== Restaurando parámetros originales ===")

params['model']['params']['n_estimators'] = 100
params['model']['params']['max_depth'] = 10

with open(project_path / "config" / "params.yaml", 'w') as f:
    yaml.dump(params, f, default_flow_style=False)

# Re-ejecutar pipeline
returncode, stdout, stderr = run_command(
    "dvc repro",
    cwd=str(project_path)
)

# Verificar que las métricas volvieron a ser iguales
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        restored_metrics = json.load(f)
    
    print("\nMétricas restauradas (deben ser iguales a las originales):")
    for metric in original_metrics:
        if metric in restored_metrics:
            match = "✓" if abs(restored_metrics[metric] - original_metrics[metric]) < 0.0001 else "✗"
            print(f"  {match} {metric}: {restored_metrics[metric]:.4f}")

print("\n✓ Reproducibilidad demostrada: mismos parámetros = mismas métricas")

## Celda 7: Ética - Trazabilidad

**Concepto:** La trazabilidad no es un lujo, es una responsabilidad. Cuando un modelo toma decisiones que afectan personas, necesitas poder responder: ¿Con qué datos se entrenó? ¿Quién lo entrenó? ¿Cómo se validó?

**Mandamiento 4:** "Datos cambian, modelos se corrompen" - Por eso necesitamos trazabilidad completa.

In [ ]:
# Implementación de ModelTracker para ética
model_tracker_script = '''
"""ModelTracker: Sistema de trazabilidad ética para modelos ML"""
import json
from datetime import datetime
from pathlib import Path
import hashlib

class ModelTracker:
    """Rastrea metadatos completos de un modelo ML para auditoría ética."""
    
    def __init__(self, model_name, version):
        self.model_name = model_name
        self.version = version
        self.metadata = {
            "model_name": model_name,
            "version": version,
            "created_at": datetime.now().isoformat(),
            "author": None,
            "description": None,
            "data_sources": [],
            "training_data": {
                "path": None,
                "version": None,
                "size": None,
                "n_samples": None,
                "date_range": None,
                "checksum": None
            },
            "preprocessing": {
                "steps": [],
                "code_version": None,
                "random_state": None
            },
            "model_config": {
                "algorithm": None,
                "hyperparameters": {},
                "features_used": [],
                "n_features": None
            },
            "evaluation": {
                "metrics": {},
                "validation_method": None,
                "test_data": None,
                "cross_validation_folds": None
            },
            "ethical_checks": {
                "bias_audit": False,
                "fairness_metrics": {},
                "approved_by": None,
                "audit_date": None,
                "concerns_raised": [],
                "mitigations_applied": []
            },
            "deployment": {
                "status": "development",
                "endpoint": None,
                "monitoring_enabled": False,
                "last_retrained": None
            },
            "audit_trail": []
        }
    
    def _add_audit_entry(self, action, details):
        """Agrega entrada al historial de auditoría."""
        self.metadata["audit_trail"].append({
            "timestamp": datetime.now().isoformat(),
            "action": action,
            "details": details
        })
    
    def set_author(self, author, email=None):
        """Establece el autor del modelo."""
        self.metadata["author"] = {
            "name": author,
            "email": email,
            "timestamp": datetime.now().isoformat()
        }
        self._add_audit_entry("set_author", f"Author: {author}")
        return self
    
    def set_data_info(self, path, version, n_samples, date_range=None, checksum=None):
        """Establece información de datos de entrenamiento."""
        self.metadata["training_data"] = {
            "path": path,
            "version": version,
            "n_samples": n_samples,
            "date_range": date_range,
            "checksum": checksum,
            "recorded_at": datetime.now().isoformat()
        }
        self.metadata["data_sources"].append({
            "path": path,
            "version": version,
            "timestamp": datetime.now().isoformat()
        })
        self._add_audit_entry("set_data_info", f"Data: {path}, version: {version}")
        return self
    
    def set_preprocessing(self, steps, code_version, random_state=None):
        """Documenta los pasos de preprocesamiento."""
        self.metadata["preprocessing"] = {
            "steps": steps,
            "code_version": code_version,
            "random_state": random_state,
            "recorded_at": datetime.now().isoformat()
        }
        self._add_audit_entry("set_preprocessing", f"Steps: {len(steps)}")
        return self
    
    def set_model_config(self, algorithm, hyperparameters, features):
        """Documenta configuración del modelo."""
        self.metadata["model_config"] = {
            "algorithm": algorithm,
            "hyperparameters": hyperparameters,
            "features_used": features,
            "n_features": len(features),
            "recorded_at": datetime.now().isoformat()
        }
        self._add_audit_entry("set_model_config", f"Algorithm: {algorithm}")
        return self
    
    def add_evaluation(self, metrics, validation_method, test_data_size=None, cv_folds=None):
        """Documenta resultados de evaluación."""
        self.metadata["evaluation"] = {
            "metrics": metrics,
            "validation_method": validation_method,
            "test_data_size": test_data_size,
            "cross_validation_folds": cv_folds,
            "recorded_at": datetime.now().isoformat()
        }
        self._add_audit_entry("add_evaluation", f"Metrics: {list(metrics.keys())}")
        return self
    
    def ethical_audit(self, bias_check_passed, fairness_metrics, approved_by, concerns=None, mitigations=None):
        """Registra auditoría ética completa."""
        self.metadata["ethical_checks"] = {
            "bias_audit": bias_check_passed,
            "fairness_metrics": fairness_metrics,
            "approved_by": approved_by,
            "audit_date": datetime.now().isoformat(),
            "concerns_raised": concerns or [],
            "mitigations_applied": mitigations or []
        }
        self._add_audit_entry("ethical_audit", f"Passed: {bias_check_passed}, Approved by: {approved_by}")
        return self
    
    def set_deployment(self, status, endpoint=None, monitoring=False):
        """Documenta estado de despliegue."""
        self.metadata["deployment"] = {
            "status": status,
            "endpoint": endpoint,
            "monitoring_enabled": monitoring,
            "last_updated": datetime.now().isoformat()
        }
        self._add_audit_entry("set_deployment", f"Status: {status}")
        return self
    
    def generate_checksum(self, data_path):
        """Genera checksum para verificar integridad de datos."""
        hasher = hashlib.md5()
        with open(data_path, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b''):
                hasher.update(chunk)
        return hasher.hexdigest()
    
    def save(self, path):
        """Guarda metadatos en archivo JSON."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        
        with open(path, 'w') as f:
            json.dump(self.metadata, f, indent=2, ensure_ascii=False)
        
        self._add_audit_entry("save_metadata", f"Path: {path}")
        print(f"Metadatos guardados en: {path}")
        return self
    
    def summary(self):
        """Imprime resumen de los metadatos."""
        print("\n" + "="*60)
        print(f"RESUMEN DEL MODELO: {self.model_name} v{self.version}")
        print("="*60)
        
        if self.metadata["author"]:
            print(f"\nAutor: {self.metadata["author"]["name"]}")
        
        print(f"\nDatos de entrenamiento:")
        print(f"  Versión: {self.metadata["training_data"]["version"]}")
        print(f"  Muestras: {self.metadata["training_data"]["n_samples"]}")
        
        print(f"\nModelo:")
        print(f"  Algoritmo: {self.metadata["model_config"]["algorithm"]}")
        print(f"  Features: {self.metadata["model_config"]["n_features"]}")
        
        print(f"\nMétricas:")
        for metric, value in self.metadata["evaluation"]["metrics"].items():
            print(f"  {metric}: {value:.4f}")
        
        print(f"\nAuditoría Ética:")
        print(f"  Sesgo verificado: {'Sí' if self.metadata['ethical_checks']['bias_audit'] else 'No'}")
        print(f"  Aprobado por: {self.metadata['ethical_checks']['approved_by']}")
        
        print(f"\nDespliegue: {self.metadata['deployment']['status']}")
        print(f"Monitoreo: {'Activado' if self.metadata['deployment']['monitoring_enabled'] else 'Desactivado'}")
        print("="*60)

def create_model_report(model_path, data_path, output_path):
    """Crea reporte de trazabilidad para un modelo existente."""
    import joblib
    import pandas as pd
    
    # Cargar modelo
    model = joblib.load(model_path)
    
    # Cargar datos para obtener información
    df = pd.read_csv(data_path)
    
    # Crear tracker
    tracker = ModelTracker(
        model_name="customer_churn_predictor",
        version="1.0.0"
    )
    
    # Llenar información
    tracker.set_author("data_science_team@empresa.com")
    tracker.set_data_info(
        path=data_path,
        version="v2",
        n_samples=len(df),
        date_range="2024-01-01 to 2024-01-31"
    )
    tracker.set_preprocessing(
        steps=[
            "Eliminar duplicados",
            "Eliminar valores nulos",
            "One-hot encoding para variables categóricas",
            "División train/test 80/20"
        ],
        code_version="v1.0",
        random_state=42
    )
    tracker.set_model_config(
        algorithm="RandomForestClassifier",
        hyperparameters=model.get_params(),
        features=list(model.feature_names_in_) if hasattr(model, 'feature_names_in_) else []
    )
    
    # Guardar
    tracker.save(output_path)
    tracker.summary()
    
    return tracker

if __name__ == '__main__':
    # Ejemplo de uso
    tracker = ModelTracker(
        model_name="customer_churn_predictor",
        version="1.0.0"
    )
    
    tracker.set_author("data_science_team@empresa.com", "ds@empresa.com")
    tracker.set_data_info(
        path="data/raw/clientes_v2.csv",
        version="v2",
        n_samples=1200,
        date_range="2024-01-01 to 2024-01-31"
    )
    tracker.set_preprocessing(
        steps=[
            "Eliminar duplicados",
            "Eliminar valores nulos",
            "One-hot encoding",
            "División 80/20"
        ],
        code_version="v1.0",
        random_state=42
    )
    tracker.set_model_config(
        algorithm="RandomForestClassifier",
        hyperparameters={"n_estimators": 100, "max_depth": 10},
        features=["age", "income", "tenure_months", "usage_score", "gender_M", "region_Norte"]
    )
    tracker.add_evaluation(
        metrics={"accuracy": 0.85, "f1": 0.82},
        validation_method="5-fold stratified CV",
        test_data_size=240,
        cv_folds=5
    )
    tracker.ethical_audit(
        bias_check_passed=True,
        fairness_metrics={
            "gender_parity": 0.95,
            "age_fairness": 0.92,
            "income_fairness": 0.88
        },
        approved_by="ethics_board@empresa.com",
        concerns=["Posible sesgo en columna income"],
        mitigations=["Monitoreo continuo de predicciones por grupo"]
    )
    tracker.set_deployment(
        status="staging",
        endpoint="https://api.empresa.com/ml/churn",
        monitoring=True
    )
    
    tracker.save("metadata/model_metadata.json")
    tracker.summary()
'''

# Guardar script del tracker
tracker_script_path = project_path / "src" / "experiments" / "model_tracker.py"
tracker_script_path.write_text(model_tracker_script)
print(f"✓ ModelTracker script creado: {tracker_script_path}")

In [ ]:
# Ejecutar tracking ético
print("=== Generando Metadatos Éticos del Modelo ===")

# Ejecutar script de tracking
returncode, stdout, stderr = run_command(
    "python src/experiments/model_tracker.py",
    cwd=str(project_path)
)

if returncode == 0:
    print(stdout)
else:
    print(f"Error: {stderr}")
    print("Salida:")
    print(stdout)

In [ ]:
# Ver metadatos generados
metadata_path = project_path / "metadata" / "model_metadata.json"

if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print("=== Metadatos del Modelo ===")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))
else:
    print("Archivo de metadatos no encontrado")

In [ ]:
# Checklist de Ética (representación visual)
print("="*60)
print("CHECKLIST DE REVISIÓN ÉTICA PARA MODELOS")
print("="*60)

checklist = {
    "Datos": [
        "¿Los datos representan adecuadamente a la población objetivo?",
        "¿Hay sesgos conocidos en los datos de entrenamiento?",
        "¿Se obtuvo consentimiento para el uso de datos personales?",
        "¿Los datos están anonimizados cuando es necesario?"
    ],
    "Modelo": [
        "¿El modelo es explicable/interpretable?",
        "¿Se realizaron pruebas de equidad (fairness)?",
        "¿El rendimiento es consistente entre subgrupos?",
        "¿Hay monitoreo continuo post-despliegue?"
    ],
    "Proceso": [
        "¿Todos los pasos están documentados?",
        "¿El pipeline es reproducible?",
        "¿Hay auditoría externa programada?",
        "¿Está claro quién es responsable del modelo?"
    ]
}

for category, items in checklist.items():
    print(f"\n{category}:")
    for item in items:
        # Simular verificación (en producción, esto sería interactivo)
        checked = "[✓]" if np.random.random() > 0.2 else "[ ]"
        print(f"  {checked} {item}")

print("\n" + "="*60)
print("Nota: Este checklist debe completarse para cada modelo en producción")
print("="*60)

In [ ]:
# Commit final a Git
print("=== Realizando Commit Final ===")

# Agregar archivos al staging (excluyendo datos que están en DVC)
run_command(
    "git add .",
    cwd=str(project_path)
)

# Ver estado
returncode, stdout, stderr = run_command(
    "git status",
    cwd=str(project_path)
)
print("Estado del repositorio:")
print(stdout)

# Commit
run_command(
    'git commit -m "feat: estructura completa del proyecto con DVC y MLflow"',
    cwd=str(project_path)
)

# Push datos a DVC remote
print("\nPush de datos a DVC remote:")
returncode, stdout, stderr = run_command(
    "dvc push",
    cwd=str(project_path)
)
if returncode == 0:
    print("✓ Datos enviados al remote storage")
else:
    print(f"Error: {stderr}")

# Mostrar historial de commits
print("\nHistorial de commits:")
returncode, stdout, stderr = run_command(
    "git log --oneline",
    cwd=str(project_path)
)
print(stdout)

print("\n" + "="*60)
print("✓ PROYECTO COMPLETADO CON ÉXITO")
print("="*60)
print("\nArchivos creados:")
print("  - Código versionado en Git")
print("  - Datos versionados en DVC")
print("  - Pipeline reproducible (dvc.yaml)")
print("  - Experimentos registrados en MLflow")
print("  - Metadatos éticos del modelo")
print("\nPara ver MLflow UI: mlflow ui")
print("Para reproducir pipeline: dvc repro")
print("Para recuperar datos: dvc pull")